runpod/pytorch:2.2.1-py3.10-cuda12.1.1-devel-ubuntu22.04

pip uninstall -y transformers accelerate bitsandbytes peft trl

pip install transformers==4.45.2 accelerate==0.34.2 bitsandbytes==0.43.3 peft==0.12.0 trl==0.11.1

pip install tiktoken einops flash-attn==2.6.3

pip install rich

In [4]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import os
from huggingface_hub import login

print("✅ 라이브러리 로드 완료!")

✅ 라이브러리 로드 완료!


In [ ]:
login(token="키")

In [2]:
# 1. 모델 ID를 EXAONE 3.0으로 변경
model_id = "LGAI-EXAONE/EXAONE-3.0-7.8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    # 2. 4090 환경의 bf16 세팅과 통일하여 연산 안정성 확보
    bnb_4bit_compute_dtype=torch.bfloat16 
)

print("📦 모델 로딩 중... (약 15GB 데이터를 GPU로 올립니다)")

📦 모델 로딩 중... (약 15GB 데이터를 GPU로 올립니다)


In [6]:
# 3. EXAONE 필수 파라미터인 trust_remote_code=True 추가
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True 
)
print("✅ 모델 로드 성공!")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

...

model-00006-of-00007.safetensors:   0%|          | 0.00/4.83G [00:00<?, ?B/s]

model-00007-of-00007.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ 모델 로드 성공!


In [7]:
# 메모리 절약을 위한 그래디언트 체크포인팅
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [8]:
# 4. 아키텍처에 맞게 타겟 모듈을 all-linear로 변경
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules="all-linear", # 개별 모듈 이름 대신 모든 선형 레이어를 타겟팅
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

print("✅ LoRA 어댑터 장착 완료!")

✅ LoRA 어댑터 장착 완료!


In [10]:
# JSONL 파일 로드
dataset = load_dataset("json", data_files={"train": "0511 train.jsonl", "val": "0511 val.jsonl"})

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

In [11]:
# Chat 템플릿 입히기
dataset = dataset.map(
    lambda x: {"text": tokenizer.apply_chat_template(x["messages"], tokenize=False)},
    remove_columns=["messages"] 
)

Map:   0%|          | 0/226 [00:00<?, ? examples/s]

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

In [12]:
# 토크나이저 패딩 토큰 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ 데이터 준비 완료! (남은 컬럼: {dataset['train'].column_names})")

✅ 데이터 준비 완료! (남은 컬럼: ['text'])


In [14]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    dataset_text_field="text",
    max_seq_length=4096, 
    args=TrainingArguments(
        output_dir="./pypi_exaone_result", # 결과 저장 폴더명 변경
        per_device_train_batch_size=1,      
        gradient_accumulation_steps=16,      
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=1,                    
        logging_first_step=True,            
        report_to="none",                   
        bf16=True,                          
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        eval_strategy="no",                 
        save_strategy="epoch",              
    ),
)

print("🔥 [초밀착 모니터링 모드] EXAONE 학습을 시작합니다!")
trainer.train()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


🔥 [초밀착 모니터링 모드] EXAONE 학습을 시작합니다!


Step,Training Loss
1,1.266000
2,1.003500
3,0.948300
4,0.761200
5,0.856300
6,0.767200
7,0.826300
8,0.776900
9,0.729800
10,0.711100


TrainOutput(global_step=42, training_loss=0.6811850397359758, metrics={'train_runtime': 1138.6386, 'train_samples_per_second': 0.595, 'train_steps_per_second': 0.037, 'total_flos': 6.854679247424717e+16, 'train_loss': 0.6811850397359758, 'epoch': 2.9734513274336285})

In [15]:
import torch

# 1. 모델을 평가 모드로 전환 (학습 중단 및 드롭아웃 비활성화)
model.eval()

def test_ai_examiner(claim_text):
    # 2. 테스트용 메시지 구성 (학습 때와 동일한 형식)
    messages = [
        {"role": "system", "content": "당신은 대한민국 특허청의 베테랑 심사관입니다. 입력된 청구항의 기재불비 여부를 논리적으로 심사하여 답변하십시오."},
        {"role": "user", "content": f"다음 청구항을 심사하여 기재불비 사항이 있다면 지적해 주세요:\n\n{claim_text}"}
    ]
    
    # 3. 템플릿 적용 및 토크나이징
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # 💡 수정 1: "cuda" 대신 model.device 사용
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device) 
    
    # 4. 답변 생성 (추론)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.7,    # 약간의 창의성 (너무 낮으면 기계적, 너무 높으면 헛소리)
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id  # 💡 수정 2: 경고 메시지 방지용 패딩 토큰 명시
        )
    
    # 5. 결과 출력 (입력 프롬프트 길이를 잘라내고 순수 생성된 텍스트만 추출)
    response = tokenizer.decode(outputs[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)
    print("\n" + "="*50)
    print(f"🔎 [심사 대상 청구항]:\n{claim_text}")
    print("-" * 50)
    print(f"🤖 [AI 심사 결과]:\n{response}")
    print("="*50 + "\n")

In [16]:
# --- 여기서 테스트하고 싶은 청구항을 입력하세요! ---
test_claim = """청구항 1 
하나 이상의 컴퓨터 및 명령어를 저장하는 하나 이상의 저장 장치를 포함하는 시스템으로서, 상기 명령어는 하
나 이상의 컴퓨터에 의해 실행될 때, 하나 이상의 컴퓨터로 하여금 입력 시퀀스를 수신하고 그리고 상기 입력
시퀀스를 프로세싱하여 출력을 생성하도록 구성된 어텐션 신경망을 구현하게 하며, 상기 어텐션 신경망은, 
어텐션 블록 입력으로부터 도출되는 쿼리 입력, 키 입력 및 값 입력을 수신하도록 구성된 어텐션 블록을 포함하
며, 상기 어텐션 블록은,
어텐션 신경망 계층 -상기 어텐션 신경망 계층은 
 쿼리 입력, 키 입력 및 값 입력에서 도출된 어텐션 계층 입력을 수신하고, 그리고
 어텐션 신경망 계층에 대한 어텐션 계층 출력을 생성하기 위해 어텐션 계층 입력에 어텐션 메커니즘을 적용하
도록 구성됨-; 그리고
게이팅 신경망 계층을 포함하며, 상기 게이팅 신경망 계층은 어텐션 신경망 계층의 어텐션 계층 출력 및 어텐션
블록 입력에 게이팅 메커니즘을 적용하여 게이팅된 어텐션 출력을 생성하도록 구성되는 것을 특징으로 하는 시
스템.
청구항 2 
제1항에 있어서, 상기 어텐션 블록은, 
계층 정규화 오퍼레이션을 쿼리 입력, 키 입력, 및 값 입력에 적용하여 정규화된 쿼리 입력, 정규화된 키 입력,
및 정규화된 값 입력을 생성하도록 구성된 제1 계층 정규화 계층을 더 포함하고, 
상기 어텐션 계층 입력은 정규화된 쿼리 입력, 정규화된 키 입력, 및 정규화된 값 입력을 포함하는 것을 특징으
로 하는 시스템.
청구항 3 
제1항 또는 제2항에 있어서, 상기 어텐션 블록 입력 및 상기 어텐션 계층 출력에 대해 게이팅 메커니즘을 적용
하는 단계는,
어텐션 블록 입력에 시그모이드 변조를 적용하여 제1 시그모이드 변조된 출력을 생성하는 단계; 그리고
상기 제1 시그모이드 변조된 출력을 상기 어텐션 계층 출력과 결합하여 게이팅된 어텐션 출력을 생성하는 단계
를 포함하는 것을 특징으로 하는 시스템.
청구항 4 
제1항 또는 제2항에 있어서, 상기 어텐션 블록 입력 및 상기 어텐션 계층 출력에 대해 게이팅 메커니즘을 적용
하는 단계는,
어텐션 계층 출력에 시그모이드 변조를 적용하여 제2 시그모이드 변조된 출력을 생성하는 단계, 그리고
제2 시그모이드 변조된 출력을 어텐션 블록 입력과 결합하여 게이팅된 어텐션 출력을 생성하는 단계를 포함하는
것을 특징으로 하는 시스템.
청구항 5 
제1항 또는 제2항에 있어서, 상기 어텐션 블록 입력 및 상기 어텐션 계층 출력에 대해 게이팅 메커니즘을 적용
하는 단계는,
시그모이드 가중치(weighting)를 사용하여 어텐션 블록 입력과 어텐션 계층 출력의 컨벡스(convex) 조합을 계산
하여 게이팅된 어텐션 출력을 생성하는 단계를 포함하는 것을 특징으로 하는 시스템.
청구항 6 
제1항 또는 제2항에 있어서, 상기 어텐션 블록 입력 및 상기 어텐션 계층 출력에 대해 게이팅 메커니즘을 적용
하는 단계는,
어텐션 계층 출력에 시그모이드 및 하이퍼볼릭탄젠트(tanh) 활성화를 적용하여 시그모이드-tanh 출력을 생성하
는 단계, 그리고
상기 시그모이드-tanh 출력을 상기 어텐션 블록 입력과 결합하여 게이팅된 어텐션 출력을 생성하는 단계를 포함
하는 것을 특징으로 하는 시스템.
청구항 7 
제1항 또는 제2항에 있어서, 상기 어텐션 블록 입력 및 상기 어텐션 계층 출력에 대해 게이팅 메커니즘을 적용
하는 단계는,
상기 어텐션 블록 입력과 어텐션 계층 출력에 게이팅된 순환 유닛을 적용하는 단계를 포함하는 것을 특징으로
하는 시스템.
청구항 8 
제1항 내지 제7항 중 어느 한 항에 있어서, 상기 어텐션 블록은,
상기 게이팅된 어텐션 출력에 계층 정규화 오퍼레이션을 적용하여 정규화된-게이팅된 어텐션 출력을 생성하도록
구성된 제2 계층 정규화 계층,
정규화된-게이팅된 어텐션 출력에 하나 이상의 변환을 적용하여 템포러리(temporary) 어텐션 블록 출력을 생성
하도록 구성된 하나 이상의 피드포워드 신경망 계층, 그리고
제2 게이팅 메커니즘을 상기 템포러리 어텐션 블록 출력 및 상기 게이팅된 어텐션 출력에 적용하여 상기 어텐션
블록에 대한 최종 어텐션 블록 출력을 생성하도록 구성된 제2 게이팅 신경망 계층을 더 포함하는 것을 특징으로
하는 시스템.
청구항 9 
제1항 내지 제8항 중 어느 한 항에 있어서, 상기 어텐션 메커니즘은 셀프-어텐션(self-attention) 메커니즘인
것을 특징으로 하는 시스템.
청구항 10 
제1항 내지 제8항 중 어느 한 항에 있어서, 상기 어텐션 메커니즘은 마스킹된 셀프-어텐션 메커니즘인 것을 특
징으로 하는 시스템.
청구항 11 
하나 이상의 컴퓨터에 의해 실행될 때, 상기 하나 이상의 컴퓨터로 하여금 제1항 내지 제10항 중 어느 한 항의
어텐션 신경망을 구현하게 하는 명령어를 저장하는 하나 이상의 컴퓨터 저장 매체. 
청구항 12 
제1항 내지 제10항 중 어느 한 항의 어텐션 신경망이 수행하도록 구성된 동작을 포함하는 방법."""

In [18]:
test_ai_examiner(test_claim)


🔎 [심사 대상 청구항]:
청구항 1 
하나 이상의 컴퓨터 및 명령어를 저장하는 하나 이상의 저장 장치를 포함하는 시스템으로서, 상기 명령어는 하
나 이상의 컴퓨터에 의해 실행될 때, 하나 이상의 컴퓨터로 하여금 입력 시퀀스를 수신하고 그리고 상기 입력
시퀀스를 프로세싱하여 출력을 생성하도록 구성된 어텐션 신경망을 구현하게 하며, 상기 어텐션 신경망은, 
어텐션 블록 입력으로부터 도출되는 쿼리 입력, 키 입력 및 값 입력을 수신하도록 구성된 어텐션 블록을 포함하
며, 상기 어텐션 블록은,
어텐션 신경망 계층 -상기 어텐션 신경망 계층은 
 쿼리 입력, 키 입력 및 값 입력에서 도출된 어텐션 계층 입력을 수신하고, 그리고
 어텐션 신경망 계층에 대한 어텐션 계층 출력을 생성하기 위해 어텐션 계층 입력에 어텐션 메커니즘을 적용하
도록 구성됨-; 그리고
게이팅 신경망 계층을 포함하며, 상기 게이팅 신경망 계층은 어텐션 신경망 계층의 어텐션 계층 출력 및 어텐션
블록 입력에 게이팅 메커니즘을 적용하여 게이팅된 어텐션 출력을 생성하도록 구성되는 것을 특징으로 하는 시
스템.
청구항 2 
제1항에 있어서, 상기 어텐션 블록은, 
계층 정규화 오퍼레이션을 쿼리 입력, 키 입력, 및 값 입력에 적용하여 정규화된 쿼리 입력, 정규화된 키 입력,
및 정규화된 값 입력을 생성하도록 구성된 제1 계층 정규화 계층을 더 포함하고, 
상기 어텐션 계층 입력은 정규화된 쿼리 입력, 정규화된 키 입력, 및 정규화된 값 입력을 포함하는 것을 특징으
로 하는 시스템.
청구항 3 
제1항 또는 제2항에 있어서, 상기 어텐션 블록 입력 및 상기 어텐션 계층 출력에 대해 게이팅 메커니즘을 적용
하는 단계는,
어텐션 블록 입력에 시그모이드 변조를 적용하여 제1 시그모이드 변조된 출력을 생성하는 단계; 그리고
상기 제1 시그모이드 변조된 출력을 상기 어텐션 계층 출력과 결합하여 게이팅된 어텐션 출력을 생성하는 단계
를 포함하는 것을 특징으로 하는 시스템.
청구항 4 
제1항 또는 제2항에 있어서, 

In [19]:
test_claim = """청구항 1 \n인공지능을 이용한 미술 자산 가치 평가 시스템에 있어서,\n인스트럭션들을 저장하는 메모리,\n프로세서를 포함하고,\n상기 인스트럭션들은, 상기 프로세서에 의해 실행 시에, 상기 시스템이\n복수의 데이터 소스로부터 미술 작품 관련 데이터를 수집하고,\n수집된 데이터를 이용하여 가치 평가 모델을 학습하며,\n학습된 모델을 이용하여 미술 작품의 가치를 평가하고,\n평가 결과를 생성하도록 제어하고,\n상기 프로세서에 의해 실행 시에, 상기 시스템이\n학습 데이터셋의 각 샘플에 대해 초기 가중치를 할당하여 제1 가중치 벡터를 생성하고, 이를 이용하여 제1 가중\n치 처리 샘플을 생성하여 저장하며,\n정보 이득율 기준의 최적 분할 특징을 선택하여 제1 의사결정 트리 모델을 학습시키고, 예측값과 실제 가치의\n차이를 계산하여 제1 전체 오차를 산출하고,\n산출된 예측 오차를 기반으로 제2 가중치 벡터를 생성하고, 이를 이용하여 제2 의사결정 트리 모델을 학습시켜\n제2 전체 오차를 산출하며,\n오차 감소율이 임계값 이하가 될 때까지 반복 학습을 수행하여 복수의 의사결정 트리 모델과 각각의 전체 오차\n를 획득하고,\n각 모델의 오차를 기반으로 기여도 계수를 산출하여 의사결정 트리 모델들을 선형 결합하며,\n예측 안정성 향상을 위해 지수 이동 평균을 적용하고 신뢰구간을 계산하여,\n평가 대상 미술 작품에 대한 최종 가치 평가값과 신뢰구간을 산출하도록 제어하는 미술 자산 가치 평가 시스템.\n청구항 2 \n제 1항에 있어서,\n상기 미술 작품 관련 데이터는\n작품의 물리적 특성 데이터, 작가의 경력 데이터 및 시장 거래 데이터를 포함하는 미술 자산 가치 평가 시스템.\n청구항 3 \n제 1항에 있어서,\n상기 가치 평가 모델은\n현재 가치 예측 모델, 가격 변동 예측 모델 및 투자 가치 평가 모델을 포함하는 미술 자산 가치 평가 시스템.\n청구항 4 \n제 1항에 있어서,\n상기 시스템은 \n작품의 희소성 지표, 성장성 지표, 시장 인지도 지표 및 컬렉터 관심도 지표를 산출하여 투자 가치를 평가하는\n미술 자산 가치 평가 시스템.\n청구항 5 \n제 1항에 있어서,\n상기 시스템은 각 평가 모델의 신뢰도 지수를 생성하고,\n신뢰도 지수에 기반하여 가중치를 적용한 통합 평가 점수를 산출하는 미술 자산 가치 평가 시스템.\n청구항 6 \n제 1항에 있어서,\n상기 시스템은 작품의 이미지를 분석하여\n색상 평가 지수, 시각적 효과 지수, 기술적 완성도 지수 및 공간감 지수를 산출하는 미술 자산 가치 평가 시스\n템.\n청구항 7 \n제 1항에 있어서,\n상기 시스템은 작품의 2차원 이미지로부터 3차원 모델을 생성하고,\n생성된 3차원 모델을 기반으로 작품의 특징을 분석하여 평가하는 미술 자산 가치 평가 시스템.\n청구항 8 \n제 1항에 있어서,\n상기 시스템은 도메인 적응 기법을 이용하여\n소스 도메인의 데이터를 타겟 도메인으로 전이하고,\n전이된 데이터를 이용하여 가치 평가 모델을 학습하는 미술 자산 가치 평가 시스템.\n청구항 9 \n제 1항에 있어서,\n상기 시스템은 시계열 데이터베이스를 구축하고,\n시장 변동 패턴, 감성 분석 결과 및 품질 지수를 기반으로 미술품의 가치 변동을 예측하는 미술 자산 가치 평가\n시스템."""

In [20]:
test_ai_examiner(test_claim)


🔎 [심사 대상 청구항]:
청구항 1 
인공지능을 이용한 미술 자산 가치 평가 시스템에 있어서,
인스트럭션들을 저장하는 메모리,
프로세서를 포함하고,
상기 인스트럭션들은, 상기 프로세서에 의해 실행 시에, 상기 시스템이
복수의 데이터 소스로부터 미술 작품 관련 데이터를 수집하고,
수집된 데이터를 이용하여 가치 평가 모델을 학습하며,
학습된 모델을 이용하여 미술 작품의 가치를 평가하고,
평가 결과를 생성하도록 제어하고,
상기 프로세서에 의해 실행 시에, 상기 시스템이
학습 데이터셋의 각 샘플에 대해 초기 가중치를 할당하여 제1 가중치 벡터를 생성하고, 이를 이용하여 제1 가중
치 처리 샘플을 생성하여 저장하며,
정보 이득율 기준의 최적 분할 특징을 선택하여 제1 의사결정 트리 모델을 학습시키고, 예측값과 실제 가치의
차이를 계산하여 제1 전체 오차를 산출하고,
산출된 예측 오차를 기반으로 제2 가중치 벡터를 생성하고, 이를 이용하여 제2 의사결정 트리 모델을 학습시켜
제2 전체 오차를 산출하며,
오차 감소율이 임계값 이하가 될 때까지 반복 학습을 수행하여 복수의 의사결정 트리 모델과 각각의 전체 오차
를 획득하고,
각 모델의 오차를 기반으로 기여도 계수를 산출하여 의사결정 트리 모델들을 선형 결합하며,
예측 안정성 향상을 위해 지수 이동 평균을 적용하고 신뢰구간을 계산하여,
평가 대상 미술 작품에 대한 최종 가치 평가값과 신뢰구간을 산출하도록 제어하는 미술 자산 가치 평가 시스템.
청구항 2 
제 1항에 있어서,
상기 미술 작품 관련 데이터는
작품의 물리적 특성 데이터, 작가의 경력 데이터 및 시장 거래 데이터를 포함하는 미술 자산 가치 평가 시스템.
청구항 3 
제 1항에 있어서,
상기 가치 평가 모델은
현재 가치 예측 모델, 가격 변동 예측 모델 및 투자 가치 평가 모델을 포함하는 미술 자산 가치 평가 시스템.
청구항 4 
제 1항에 있어서,
상기 시스템은 
작품의 희소성 지표, 성장성 지표, 시장 인지도 지표 및 컬렉터 관심도 지표를 산출하여 투자 가치를

In [21]:
import shutil

# 1. 저장 경로 설정 (EXAONE 모델임을 명시)
save_path = "pypi_exaone_examiner_lora"

# 2. 모델(LoRA 어댑터 가중치) 및 토크나이저 저장
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"🎉 성공! 학습된 EXAONE 심사관 모델이 '{save_path}' 폴더에 안전하게 저장되었습니다.")
print("이제 이 폴더(어댑터)와 원본 모델만 있으면 언제든 AI 심사관을 소환할 수 있습니다.")

# 3. 학습된 폴더를 zip 파일로 압축 (다운로드를 위해)
zip_filename = "pypi_exaone_model"
shutil.make_archive(zip_filename, 'zip', save_path)

print(f"✅ 압축 완료! 이제 왼쪽 파일 탐색기(또는 Jupyter 트리)에서 '{zip_filename}.zip'을 찾아 로컬 PC로 다운로드하세요.")

🎉 성공! 학습된 EXAONE 심사관 모델이 'pypi_exaone_examiner_lora' 폴더에 안전하게 저장되었습니다.
이제 이 폴더(어댑터)와 원본 모델만 있으면 언제든 AI 심사관을 소환할 수 있습니다.
✅ 압축 완료! 이제 왼쪽 파일 탐색기(또는 Jupyter 트리)에서 'pypi_exaone_model.zip'을 찾아 로컬 PC로 다운로드하세요.
